In [6]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [7]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

rogii_wellbore_geology_prediction_path = kagglehub.competition_download('rogii-wellbore-geology-prediction')

print('Data source import complete.')


Data source import complete.


# [9.946 PB] Geostatistics, Softmax NCC & Physics Hybrid

[ play off of Geostats, Softmax & Physics NCC] added in:


INSPIRED BY:: '/kaggle/input/competitions/rogii-wellbore-geology-prediction'. this is Version 2.3.

(never rana 63 minute training data before.)

by : Sanidhya Vijay24


I added in extra regressor layers, and a KNN model layer, and I added in Softmax Physics Post-Processing.
also added in CNN/KNN snapshot layer.


        So CatBoostRegressor and GroupKFold were given extra attention.

[]

This notebook pushes past the 10.0 barrier by combining spatial geostatistics, multi-scale signal matching, and a physics-based particle filter directly into the ensemble pipeline.

If this notebook helps improve your leaderboard score, an upvote would be greatly appreciated.

## Core Methodology & Innovations

### 1. Segmented per-Formation `b_well` Features

Instead of using a single global `b_well` offset, this approach computes 5 different offsets:

- `b_full`
- `b_early`
- `b_mid`
- `b_late`
- `b_wls`

These are calculated across all 6 geological formations.

The goal was to capture how the structural dip changes over different drilling phases rather than treating it as one static behavior. These features give the boosting models a much richer spatial understanding of the formations.

---

### 2. Multi-Scale NCC with Softmax Ensembling

Rather than relying on a single Gamma Ray alignment window, the notebook runs Normalized Cross-Correlation (NCC) at 3 different scales:

- 8
- 15
- 25

The predictions from each window are then combined using a Softmax-weighted consensus (`exp(3*scores)`), producing a more stable and reliable `sc_ens` prediction.

This helps reduce noisy shifts while still preserving local alignment behavior.

---

### 3. Numba-Accelerated Particle Filters

Two Monte Carlo Particle Filters are used:

- one for predicting `ANCC` depth
- another for tracking geological velocity `Z`

They are implemented using `numba @njit`, allowing them to run extremely fast while still modeling the physical constraints of the drilling process.

The objective here was to inject actual physical behavior into the pipeline instead of relying purely on ML correlations.

---

### 4. Ridge Stacking + Physics Hybrid Post-Processing

The final ensemble stacks LightGBM, XGBoost, and CatBoost using a positive-weight Ridge regressor for more stable blending.

After that, a grid-search over OOF predictions tunes variables such as:

- `alpha`
- `tau`
- `w_pf`

The key variable is `w_pf`, which directly blends the Particle Filter prediction into the final output.

This makes the final prediction pipeline a genuine Physics + ML hybrid system rather than just a standard ensemble of ML models.

-

In [15]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
from scipy.optimize import minimize
from scipy.optimize import minimize
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from sklearn.model_selection import GroupKFold
from numba import njit
import warnings
warnings.filterwarnings("ignore")

# Correcting DATA_DIR to point to the downloaded Kaggle data
# The 'rogii_wellbore_geology_prediction_path' variable should directly point to the competition data root.
DATA_DIR = Path(rogii_wellbore_geology_prediction_path)

OUTPUT_PATH = Path('/kaggle/working/submission.csv')
if not Path('/kaggle/working').exists():
    OUTPUT_PATH = Path('submission.csv')

N_SPLITS = 3
SPLIT_SEED = 42
FORMATIONS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PLANE_K = 10
ROW_K = 20
ROW_NQ = 12_000
OFFSETS = np.array([-80, -40, -20, -10, -5, 0, 5, 10, 20, 40, 80], dtype=np.float32)

In [9]:
# Particle Filter Constants
PF_N_PARTICLES = 500
PF_MOMENTUM_ALPHA = 0.993
PF_Z_SIGMA_FLOOR = 0.005
PF_Z_SIGMA_SCALE = 2.0
PF_VELOCITY_NOISE_STD = 0.005
PF_POSITION_NOISE_STD = 0.01
PF_INIT_VELOCITY_STD = 0.02
PF_GR_SIGMA_MIN = 10.0
PF_GR_SIGMA_MAX = 60.0
PF_GR_SIGMA_DEFAULT = 30.0
PF_INIT_SPREAD_STD = 0.5
PF_RESAMPLE_THRESHOLD = 0.5
PF_ROUGHENING_STD_POS = 0.2
PF_ROUGHENING_STD_VEL = 0.003
PF_GR_ROLLING_WINDOW = 5
PF_GR_ROLLING_WEIGHT = 0.3

ANCC_ALPHA = 0.998
ANCC_RATE_NOISE_STD = 0.002
ANCC_POS_NOISE_STD = 0.005
ANCC_INIT_RATE_STD = 0.01
ANCC_INIT_SPREAD_STD = 0.3
ANCC_ROUGHENING_STD_POS = 0.1
ANCC_ROUGHENING_STD_RATE = 0.001
ANCC_N_PARTICLES = 500

DENSE_SAMPLES_PER_WELL = 40
DENSE_K = 20

print('DATA_DIR =', DATA_DIR)
print('OUTPUT_PATH =', OUTPUT_PATH)

DATA_DIR = /kaggle/input/rogii-wellbore-geology-prediction
OUTPUT_PATH = submission.csv


In [10]:
# Helper functions
def recent_mean_diff(values: np.ndarray, window: int) -> float:
    values = values[-(window + 1):]
    if len(values) < 2: return 0.0
    return float(np.diff(values).mean())

def recent_slope(y_values: np.ndarray, x_values: np.ndarray, window: int) -> float:
    y_values = y_values[-window:]
    x_values = x_values[-window:]
    if len(y_values) < 2: return 0.0
    centered_x = x_values - x_values.mean()
    denominator = float(np.dot(centered_x, centered_x))
    if denominator == 0.0: return 0.0
    return float(np.dot(centered_x, y_values - y_values.mean()) / denominator)

def nearest_index(sorted_values: np.ndarray, target: float) -> int:
    idx = int(np.searchsorted(sorted_values, target, side='left'))
    if idx >= len(sorted_values): return len(sorted_values) - 1
    if idx > 0 and abs(sorted_values[idx - 1] - target) <= abs(sorted_values[idx] - target):
        return idx - 1
    return idx

def fill_and_smooth_gr(values: np.ndarray, fallback: float, radius: int) -> np.ndarray:
    series = pd.Series(values, dtype='float32').interpolate(limit_direction='both').fillna(fallback)
    if radius <= 0: return series.to_numpy(dtype=np.float32)
    return series.rolling(radius * 2 + 1, center=True, min_periods=1).mean().to_numpy(dtype=np.float32)

def interp_sorted(x: np.ndarray, y: np.ndarray, target: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    target = np.asarray(target, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2: return np.full(len(target), np.nan)
    xs = x[mask]
    ys = y[mask]
    order = np.argsort(xs)
    xs = xs[order]
    ys = ys[order]
    xs, idx = np.unique(xs, return_index=True)
    ys = ys[idx]
    if len(xs) < 2: return np.full(len(target), np.nan)
    return np.interp(target, xs, ys, left=np.nan, right=np.nan)

def calc_wls_b_well(b_vals: np.ndarray, decay: float = 0.02) -> float:
    n = len(b_vals)
    if n == 0: return 0.0
    w = np.exp(-decay * np.arange(n - 1, -1, -1))
    return float(np.sum(w * b_vals) / np.sum(w))

def multi_scale_ncc(kgr, ktvt, hgr, hws=(8,15,25), stride=3):
    out = []
    for hw in hws:
        win = 2 * hw + 1; nk = len(kgr); nh = len(hgr)
        if nk < win + 1 or nh == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32)))
            continue
        kg = pd.Series(kgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        hg = pd.Series(hgr).rolling(5, center=True, min_periods=1).mean().values.astype(np.float32)
        sts = np.arange(0, nk - win + 1, stride, dtype=np.int32); M = len(sts)
        if M == 0:
            out.append((np.full(nh, ktvt[-1], np.float32), np.zeros(nh, np.float32)))
            continue
        C = kg[sts[:, None] + np.arange(win, dtype=np.int32)[None, :]].astype(np.float32)
        Cn = (C - C.mean(1, keepdims=True)) / (C.std(1, keepdims=True) + 1e-6)
        hp = np.pad(hg, hw, mode='edge')
        H = hp[np.arange(nh)[:, None] + np.arange(win)[None, :]].astype(np.float32)
        Hn = (H - H.mean(1, keepdims=True)) / (H.std(1, keepdims=True) + 1e-6)
        ncc = Hn @ Cn.T / win
        best = ncc.argmax(1)
        score = ncc.max(1).astype(np.float32)
        out.append((ktvt[np.clip(sts[best] + hw, 0, nk - 1)].astype(np.float32), score))

    tvts = np.stack([o[0] for o in out], 1)
    scores = np.stack([o[1] for o in out], 1)
    sw = np.exp(3. * scores)
    sw /= sw.sum(1, keepdims=True) + 1e-9
    sc_ens = (tvts * sw).sum(1).astype(np.float32)
    return out, sc_ens

def seg_b_well(ktvt, kz, form_col):
    bv = ktvt + kz - form_col
    n = len(bv)
    b_full = float(np.median(bv))
    b_late = float(np.median(bv[max(0, n - 50):])) if n >= 5 else b_full
    t1, t2 = n // 3, 2 * n // 3
    b_early = float(np.median(bv[:max(1, t1)])) if t1 > 0 else b_full
    b_mid = float(np.median(bv[t1:max(t1 + 1, t2)])) if t2 > t1 else b_full
    w = np.exp(0.02 * np.arange(n))
    w /= w.sum()
    b_wls = float(np.dot(w, bv))
    return b_full, b_early, b_mid, b_late, b_wls

def visible_gr_shift_fit(known_tvt: np.ndarray, known_gr: np.ndarray, tw_tvt: np.ndarray, tw_gr: np.ndarray) -> dict[str, float]:
    out = {"visible_gr_shift_ft": 0.0, "visible_gr_shift_corr": 0.0, "visible_gr_bias": 0.0}
    known_tvt = np.asarray(known_tvt, dtype=float)
    known_gr = np.asarray(known_gr, dtype=float)
    if len(known_tvt) < 50 or len(tw_tvt) < 10: return out
    best_corr = -np.inf
    best_shift = 0.0
    for shift in np.arange(-30.0, 30.1, 2.0):
        candidate = interp_sorted(tw_tvt, tw_gr, known_tvt + shift)
        mask = np.isfinite(candidate) & np.isfinite(known_gr)
        if mask.sum() < 30 or np.nanstd(candidate[mask]) <= 1e-6 or np.nanstd(known_gr[mask]) <= 1e-6:
            continue
        corr = float(np.corrcoef(known_gr[mask], candidate[mask])[0, 1])
        if np.isfinite(corr) and corr > best_corr:
            best_corr = corr
            best_shift = float(shift)
    if not np.isfinite(best_corr): return out
    candidate = interp_sorted(tw_tvt, tw_gr, known_tvt + best_shift)
    mask = np.isfinite(candidate) & np.isfinite(known_gr)
    out["visible_gr_shift_ft"] = best_shift
    out["visible_gr_shift_corr"] = best_corr
    out["visible_gr_bias"] = float(np.nanmean(known_gr[mask] - candidate[mask])) if mask.any() else 0.0
    return out

In [11]:
# Beam Search Algorithm
@njit
def beam_predict_numba(smoothed, tw_tvt_len, tw_gr, start_idx, beam_size, move_cost, emit_scale):
    n_steps = len(smoothed)
    curr_states = np.zeros(1, dtype=np.int32)
    curr_states[0] = start_idx
    curr_costs = np.zeros(1, dtype=np.float32)

    backpointers = np.zeros((n_steps, tw_tvt_len), dtype=np.int32)

    for t in range(n_steps):
        gr_value = smoothed[t]
        cand_costs = np.full(tw_tvt_len, np.inf, dtype=np.float32)
        cand_parents = np.zeros(tw_tvt_len, dtype=np.int32)

        for i in range(len(curr_states)):
            idx = curr_states[i]
            cost = curr_costs[i]
            for delta in (-1, 0, 1):
                ni = idx + delta
                if 0 <= ni < tw_tvt_len:
                    emit = ((gr_value - tw_gr[ni]) ** 2) / emit_scale
                    tot = cost + emit + move_cost * abs(delta)
                    if tot < cand_costs[ni]:
                        cand_costs[ni] = tot
                        cand_parents[ni] = idx

        valid_mask = cand_costs < np.inf
        valid_indices = np.where(valid_mask)[0]
        if len(valid_indices) == 0:
            return np.zeros(0, dtype=np.int32)

        valid_costs = cand_costs[valid_indices]
        sort_idx = np.argsort(valid_costs)

        keep_n = min(beam_size, len(sort_idx))
        next_states = np.zeros(keep_n, dtype=np.int32)
        next_costs = np.zeros(keep_n, dtype=np.float32)

        for k in range(keep_n):
            best_ni = valid_indices[sort_idx[k]]
            next_states[k] = best_ni
            next_costs[k] = cand_costs[best_ni]
            backpointers[t, best_ni] = cand_parents[best_ni]

        curr_states = next_states
        curr_costs = next_costs

    path = np.zeros(n_steps, dtype=np.int32)
    final_idx = curr_states[np.argmin(curr_costs)]
    path[n_steps - 1] = final_idx
    for t in range(n_steps - 1, 0, -1):
        path[t - 1] = backpointers[t, path[t]]

    return path

def beam_predict(gr_values, tw_tvt, tw_gr, start_tvt, beam_size, move_cost, emit_scale, radius):
    start_idx = nearest_index(tw_tvt, start_tvt)
    smoothed = fill_and_smooth_gr(gr_values, float(np.nanmean(tw_gr)), radius)
    path_indices = beam_predict_numba(smoothed, len(tw_tvt), tw_gr, start_idx, beam_size, float(move_cost), float(emit_scale))
    if len(path_indices) == 0:
        return np.full(len(smoothed), tw_tvt[start_idx], dtype=np.float32)
    return tw_tvt[path_indices]

def gr_fft_features(gr_post):
    valid = gr_post[~np.isnan(gr_post)]
    if len(valid) < 32: return 0.0, 0.0
    centered = valid - valid.mean()
    spec = np.abs(np.fft.rfft(centered)) ** 2
    if len(spec) < 3: return 0.0, 0.0
    dom = int(np.argmax(spec[1:])) + 1
    return float(dom / len(valid)), float(np.log1p(spec[dom]))

class FormationPlaneKNN:
    def __init__(self, train_paths):
        rows = []
        for p in train_paths:
            wid = p.name.split("__")[0]
            try:
                df = pd.read_csv(p, usecols=["X", "Y"] + FORMATIONS).dropna()
            except Exception: continue
            if len(df) == 0: continue
            row = {"wid": wid, "x": float(df["X"].median()), "y": float(df["Y"].median())}
            for c in FORMATIONS:
                row[f"{c}_med"] = float(df[c].median())
            rows.append(row)
        self.df = pd.DataFrame(rows)
        self.wid_idx = {w: i for i, w in enumerate(self.df["wid"].to_numpy())}
        xy = self.df[["x", "y"]].to_numpy()
        self.scale = xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(xy / self.scale)
        self.x_arr = self.df["x"].to_numpy()
        self.y_arr = self.df["y"].to_numpy()
        self.formation_arr = self.df[[f"{c}_med" for c in FORMATIONS]].to_numpy(dtype=np.float64)

    def impute(self, xy_q, self_wid=None, k=PLANE_K):
        q = xy_q / self.scale
        n_q = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=n_q)
        if self_wid is not None and self_wid in self.wid_idx:
            self_i = self.wid_idx[self_wid]
            mask_self = idx == self_i
            dist = np.where(mask_self, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_q - 1), axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        w = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0).astype(np.float64)
        x_n = self.x_arr[idx_k]
        y_n = self.y_arr[idx_k]
        wx = w * x_n
        wy = w * y_n
        ATWA = np.zeros((len(xy_q), 3, 3))
        ATWA[:, 0, 0] = (wx * x_n).sum(axis=1) + 1e-9
        ATWA[:, 0, 1] = (wx * y_n).sum(axis=1)
        ATWA[:, 0, 2] = wx.sum(axis=1)
        ATWA[:, 1, 0] = ATWA[:, 0, 1]
        ATWA[:, 1, 1] = (wy * y_n).sum(axis=1) + 1e-9
        ATWA[:, 1, 2] = wy.sum(axis=1)
        ATWA[:, 2, 0] = ATWA[:, 0, 2]
        ATWA[:, 2, 1] = ATWA[:, 1, 2]
        ATWA[:, 2, 2] = w.sum(axis=1) + 1e-9

        f_n = self.formation_arr[idx_k]
        ATWb_x = ((wx[:, :, None] * f_n).sum(axis=1))
        ATWb_y = ((wy[:, :, None] * f_n).sum(axis=1))
        ATWb_c = ((w[:, :, None] * f_n).sum(axis=1))
        rhs = np.stack([ATWb_x, ATWb_y, ATWb_c], axis=1)

        try:
            coef = np.linalg.solve(ATWA, rhs)
        except np.linalg.LinAlgError:
            coef = np.zeros((len(xy_q), 3, 6))
            for r in range(len(xy_q)):
                try: coef[r] = np.linalg.pinv(ATWA[r]) @ rhs[r]
                except Exception: coef[r] = 0

        X_q = xy_q[:, 0]
        Y_q = xy_q[:, 1]
        formations = (X_q[:, None] * coef[:, 0, :] + Y_q[:, None] * coef[:, 1, :] + coef[:, 2, :]).astype(np.float32)

        no_n = (~valid_k).all(axis=1)
        if no_n.any():
            formations[no_n] = self.formation_arr.mean(axis=0)

        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1).astype(np.float32)
        return formations, min_dist

class RowKNN:
    def __init__(self, train_paths):
        xs, ys, anccs, wid_arr = [], [], [], []
        for p in train_paths:
            wid = p.name.split("__")[0]
            try: df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
            except Exception: continue
            if len(df) == 0: continue
            xs.append(df["X"].to_numpy())
            ys.append(df["Y"].to_numpy())
            anccs.append(df["ANCC"].to_numpy())
            wid_arr.extend([wid] * len(df))

        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32)
        self.wids = np.array(wid_arr)
        self.scale = self.xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=ROW_K, n_q=ROW_NQ):
        q = xy_q / self.scale
        n_q = min(n_q, len(self.ancc))
        dist, idx = self.tree.query(q, k=n_q, workers=-1)
        if self_wid is not None:
            mask_self = self.wids[idx] == self_wid
            dist = np.where(mask_self, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_q - 1), axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        w = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        sw = w.sum(axis=1)
        no_n = sw < 1e-9
        safe = np.where(no_n, 1.0, sw)
        ancc_pred = (self.ancc[idx_k] * w).sum(axis=1) / safe
        ancc_pred = np.where(no_n, float(self.ancc.mean()), ancc_pred)
        diff_sq = (self.ancc[idx_k] - ancc_pred[:, None]) ** 2
        var = (diff_sq * w).sum(axis=1) / safe
        std = np.sqrt(np.maximum(var, 0.0))
        min_dist = np.where(valid_k, d_k, np.inf).min(axis=1)
        return ancc_pred.astype(np.float32), std.astype(np.float32), min_dist.astype(np.float32)

class DenseANCCImputer:
    def __init__(self, train_paths):
        xs, ys, anccs, wid_arr = [], [], [], []
        for p in train_paths:
            wid = p.name.split("__")[0]
            try: df = pd.read_csv(p, usecols=["X", "Y", "ANCC"]).dropna()
            except Exception: continue
            if len(df) == 0: continue
            idx = np.linspace(0, len(df) - 1, min(DENSE_SAMPLES_PER_WELL, len(df)), dtype=int)
            sampled = df.iloc[idx]
            xs.append(sampled["X"].to_numpy())
            ys.append(sampled["Y"].to_numpy())
            anccs.append(sampled["ANCC"].to_numpy())
            wid_arr.extend([wid] * len(sampled))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32)
        self.wids = np.array(wid_arr)
        self.wid_set = set(wid_arr)
        self.scale = self.xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_q, self_wid=None, k=DENSE_K):
        xy_q = np.atleast_2d(xy_q)
        q = xy_q / self.scale
        n_q = min(k + (DENSE_SAMPLES_PER_WELL + 5), len(self.ancc))
        dist, idx = self.tree.query(q, k=n_q, workers=-1)
        if self_wid is not None and self_wid in self.wid_set:
            mask_self = self.wids[idx] == self_wid
            dist = np.where(mask_self, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_q - 1), axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid_k = np.isfinite(d_k)
        w = np.where(valid_k, 1.0 / (d_k + 1e-3), 0.0)
        sw = w.sum(axis=1)
        no_n = sw < 1e-9
        safe = np.where(no_n, 1.0, sw)
        ancc_n = self.ancc[idx_k]
        ancc_pred = (ancc_n * w).sum(axis=1) / safe
        ancc_pred = np.where(no_n, float(self.ancc.mean()), ancc_pred)
        diff_sq = (ancc_n - ancc_pred[:, None]) ** 2
        var = (diff_sq * w).sum(axis=1) / safe
        neighbor_std = np.sqrt(np.maximum(var, 0.0))
        d_finite = np.where(valid_k, d_k, np.inf)
        min_dist = d_finite.min(axis=1)
        return ancc_pred.astype(np.float32), min_dist.astype(np.float32), neighbor_std.astype(np.float32)

def pf_calibrate_gr_sigma(known, tw_tvt, tw_gr):
    known_gr = known[known['GR'].notna()]
    if len(known_gr) < 20: return PF_GR_SIGMA_DEFAULT
    tw_func = interp1d(tw_tvt, tw_gr, bounds_error=False, fill_value=(tw_gr[0], tw_gr[-1]))
    expected = tw_func(known_gr['TVT_input'].to_numpy())
    residuals = known_gr['GR'].to_numpy() - expected
    return float(np.clip(np.std(residuals), PF_GR_SIGMA_MIN, PF_GR_SIGMA_MAX))

def pf_estimate_init_velocity(known):
    if len(known) < 10: return 0.0
    tail = known.tail(20)
    if len(tail) < 5: return 0.0
    dtvt = np.diff(tail['TVT_input'].to_numpy())
    dmd = np.diff(tail['MD'].to_numpy())
    mask = dmd > 0
    if mask.sum() < 3: return 0.0
    return float(np.median(dtvt[mask] / dmd[mask]))

def pf_learn_z_beta(known):
    if len(known) < 30: return -1.0, 0.0, 0.1
    dz = np.diff(known['Z'].to_numpy())
    dtvt = np.diff(known['TVT_input'].to_numpy())
    dmd = np.diff(known['MD'].to_numpy())
    mask = dmd > 0
    if mask.sum() < 10: return -1.0, 0.0, 0.1
    vz = dz[mask] / dmd[mask]
    vt = dtvt[mask] / dmd[mask]
    A = np.column_stack([vz, np.ones_like(vz)])
    coef, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
    residuals = vt - (coef[0] * vz + coef[1])
    sigma = max(np.std(residuals), 0.001)
    return float(coef[0]), float(coef[1]), float(sigma)

@njit(cache=True)
def _numba_interp1(xp, fp, x):
    n = len(xp)
    if x <= xp[0]:  return fp[0]
    if x >= xp[-1]: return fp[-1]
    lo = 0; hi = n - 1
    while hi - lo > 1:
        mid = (lo + hi) >> 1
        if xp[mid] <= x: lo = mid
        else:             hi = mid
    t = (x - xp[lo]) / (xp[hi] - xp[lo])
    return fp[lo] + t * (fp[hi] - fp[lo])

@njit(cache=True)
def _pf_z_loop(md_v, z_v, gr_v, gr_smooth_v, tw_tvt, tw_gr, tw_smooth_gr, pos, vel, w, prev_md, prev_z, tvt_min, tvt_max, gr_sigma, beta, intercept, z_sigma):
    n_e = len(md_v)
    N = len(pos)
    pts = np.empty(n_e, np.float32)
    std = np.empty(n_e, np.float32)
    for i in range(n_e):
        d_md = max(md_v[i] - prev_md, 1.0)
        dz_dmd = (z_v[i] - prev_z) / d_md
        v_expected = beta * dz_dmd + intercept

        vel = 0.993 * vel + np.random.normal(0, 0.005, N)
        pos = pos + vel * d_md + np.random.normal(0, 0.01, N)
        for j in range(N):
            pos[j] = min(max(pos[j], tvt_min - 50.0), tvt_max + 50.0)

        if not np.isnan(gr_v[i]):
            for j in range(N):
                expected_point = _numba_interp1(tw_tvt, tw_gr, pos[j])
                diff_point = gr_v[i] - expected_point
                lik_point = np.exp(-0.5 * (diff_point / gr_sigma) ** 2)

                if not np.isnan(gr_smooth_v[i]):
                    expected_smooth = _numba_interp1(tw_tvt, tw_smooth_gr, pos[j])
                    diff_smooth = gr_smooth_v[i] - expected_smooth
                    lik_smooth = np.exp(-0.5 * (diff_smooth / (gr_sigma * 1.5)) ** 2)
                    lik = (1.0 - 0.3) * lik_point + 0.3 * lik_smooth
                else:
                    lik = lik_point
                w[j] *= max(lik, 1e-300)
            ws = np.sum(w)
            if ws > 0: w /= ws
            else: w[:] = 1.0 / N

        z_sig = max(z_sigma * 2.0, 0.005)
        for j in range(N):
            lik_z = np.exp(-0.5 * ((vel[j] - v_expected) / z_sig) ** 2)
            w[j] *= max(lik_z, 1e-300)
        ws = np.sum(w)
        if ws > 0: w /= ws
        else: w[:] = 1.0 / N

        ne = 1.0 / np.sum(w ** 2)
        if ne < 0.5 * N:
            cum = np.cumsum(w)
            u = (np.arange(N) + np.random.uniform()) / N
            ix = np.searchsorted(cum, u)
            pos[:] = pos[ix] + np.random.normal(0, 0.2, N)
            vel[:] = vel[ix] + np.random.normal(0, 0.003, N)
            w[:] = 1.0 / N

        mu = 0.0
        for j in range(N): mu += w[j] * pos[j]
        var = 0.0
        for j in range(N): var += w[j] * (pos[j] - mu) ** 2
        pts[i] = mu
        std[i] = np.sqrt(var)
        prev_md = md_v[i]
        prev_z = z_v[i]
    return pts, std

def run_pf_z_velocity(known, evalz, hw_gr_smooth, tw_tvt, tw_gr):
    n_particles = PF_N_PARTICLES
    tw_smooth_gr = pd.Series(tw_gr).rolling(PF_GR_ROLLING_WINDOW, center=True, min_periods=1).mean().to_numpy()
    tvt_min, tvt_max = float(tw_tvt.min()), float(tw_tvt.max())
    gr_sigma = pf_calibrate_gr_sigma(known, tw_tvt, tw_gr)
    beta, intercept, z_sigma = pf_learn_z_beta(known)
    if len(evalz) == 0: return np.array([]), np.array([])

    last_tvt = float(known['TVT_input'].iloc[-1])
    positions = last_tvt + np.random.normal(0, PF_INIT_SPREAD_STD, n_particles)
    init_v = pf_estimate_init_velocity(known)
    velocities = init_v + np.random.normal(0, PF_INIT_VELOCITY_STD, n_particles)
    weights = np.ones(n_particles) / n_particles

    md_vals = evalz['MD'].to_numpy(dtype=np.float64)
    gr_vals = evalz['GR'].to_numpy(dtype=np.float64)
    z_vals = evalz['Z'].to_numpy(dtype=np.float64)
    gr_smooth_v = hw_gr_smooth.loc[evalz.index].to_numpy(dtype=np.float64)
    prev_md = float(known['MD'].iloc[-1])
    prev_z = float(known['Z'].iloc[-1])

    pts, stds = _pf_z_loop(
        md_vals, z_vals, gr_vals, gr_smooth_v,
        tw_tvt.astype(np.float64), tw_gr.astype(np.float64), tw_smooth_gr.astype(np.float64),
        positions.astype(np.float64), velocities.astype(np.float64), weights.astype(np.float64),
        prev_md, prev_z, tvt_min, tvt_max, gr_sigma, beta, intercept, z_sigma
    )
    return pts, stds

def ancc_estimate_init_rate(known):
    if len(known) < 10: return 0.0
    tail = known.tail(30)
    dtvt = np.diff(tail['TVT_input'].to_numpy())
    dz = np.diff(tail['Z'].to_numpy())
    dmd = np.diff(tail['MD'].to_numpy())
    dancc = dtvt + dz
    mask = dmd > 0
    if mask.sum() < 3: return 0.0
    return float(np.median(dancc[mask] / dmd[mask]))

@njit(cache=True)
def _pf_ancc_loop(md_v, z_v, gr_v, tw_tvt, tw_gr, pos, rate, w, prev_md, tvt_min, tvt_max, gr_sigma):
    n_e = len(md_v)
    N = len(pos)
    pts = np.empty(n_e, np.float32)
    std = np.empty(n_e, np.float32)
    for i in range(n_e):
        d_md = max(md_v[i] - prev_md, 1.0)
        rate = 0.998 * rate + np.random.normal(0, 0.002, N)
        pos = pos + rate * d_md + np.random.normal(0, 0.005, N)
        tvt_est = pos - z_v[i]
        for j in range(N):
            tvt_est[j] = min(max(tvt_est[j], tvt_min - 50.0), tvt_max + 50.0)
            pos[j] = tvt_est[j] + z_v[i]

        if not np.isnan(gr_v[i]):
            for j in range(N):
                expected_gr = _numba_interp1(tw_tvt, tw_gr, tvt_est[j])
                lik = np.exp(-0.5 * ((gr_v[i] - expected_gr) / gr_sigma) ** 2)
                w[j] *= max(lik, 1e-300)
            ws = np.sum(w)
            if ws > 0: w /= ws
            else: w[:] = 1.0 / N

        ne = 1.0 / np.sum(w ** 2)
        if ne < 0.5 * N:
            cum = np.cumsum(w)
            u = (np.arange(N) + np.random.uniform()) / N
            ix = np.searchsorted(cum, u)
            pos[:] = pos[ix] + np.random.normal(0, 0.1, N)
            rate[:] = rate[ix] + np.random.normal(0, 0.001, N)
            w[:] = 1.0 / N

        mu = 0.0
        for j in range(N): mu += w[j] * (pos[j] - z_v[i])
        var = 0.0
        for j in range(N): var += w[j] * (pos[j] - z_v[i] - mu) ** 2
        pts[i] = mu
        std[i] = np.sqrt(var)
        prev_md = md_v[i]
    return pts, std

def run_pf_ancc(known, evalz, tw_tvt, tw_gr):
    n_particles = ANCC_N_PARTICLES
    tvt_min, tvt_max = float(tw_tvt.min()), float(tw_tvt.max())
    gr_sigma = pf_calibrate_gr_sigma(known, tw_tvt, tw_gr)
    if len(evalz) == 0: return np.array([]), np.array([])
    last_state = float(known['TVT_input'].iloc[-1]) + float(known['Z'].iloc[-1])
    init_rate = ancc_estimate_init_rate(known)
    pos = last_state + np.random.normal(0, ANCC_INIT_SPREAD_STD, n_particles)
    rate = init_rate + np.random.normal(0, ANCC_INIT_RATE_STD, n_particles)
    w = np.ones(n_particles) / n_particles

    md_vals = evalz['MD'].to_numpy(dtype=np.float64)
    z_vals = evalz['Z'].to_numpy(dtype=np.float64)
    gr_vals = evalz['GR'].to_numpy(dtype=np.float64)
    prev_md = float(known['MD'].iloc[-1])

    pts, stds = _pf_ancc_loop(
        md_vals, z_vals, gr_vals,
        tw_tvt.astype(np.float64), tw_gr.astype(np.float64),
        pos.astype(np.float64), rate.astype(np.float64), w.astype(np.float64),
        prev_md, tvt_min, tvt_max, gr_sigma
    )
    return pts, stds

def build_hidden_features(horizontal_path: Path, typewell_path: Path, is_train: bool, formation_imputer, row_imputer, dense_imputer) -> pd.DataFrame | None:
    wid = horizontal_path.name.split('__')[0]
    df = pd.read_csv(horizontal_path)
    mask = df['TVT_input'].isna().to_numpy()
    if not mask.any(): return None

    mask_start = int(np.flatnonzero(mask)[0])
    if mask_start == 0: return None

    known = df.iloc[:mask_start].copy()
    hidden = df.iloc[mask_start:].copy()
    last_known = known.iloc[-1]

    tw = pd.read_csv(typewell_path) if typewell_path.exists() else None

    if tw is not None and not tw.empty:
        tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
        tw_gr = tw['GR'].to_numpy(dtype=np.float32)
    else:
        if is_train: return None
        else:
            tw_tvt = np.array([0.0], dtype=np.float32)
            tw_gr = np.array([0.0], dtype=np.float32)

    gr_full = df['GR'].interpolate(limit_direction='both')
    if gr_full.isna().any():
        gr_full = gr_full.fillna(float(np.nanmean(tw_gr)))

    gr_roll5 = gr_full.rolling(5, center=True, min_periods=1).mean()
    gr_roll21 = gr_full.rolling(21, center=True, min_periods=1).mean()
    gr_grad = gr_full.diff().fillna(0.0)
    gr_std5 = gr_full.rolling(5, center=True, min_periods=1).std().fillna(0.0)
    gr_std21 = gr_full.rolling(21, center=True, min_periods=1).std().fillna(0.0)
    gr_lag1 = gr_full.shift(1).bfill()
    gr_lead1 = gr_full.shift(-1).ffill()
    gr_lag5 = gr_full.shift(5).bfill()
    gr_lead5 = gr_full.shift(-5).ffill()
    gr_cumsum = gr_full.cumsum()

    known_tvt = known['TVT_input'].to_numpy(dtype=np.float32)
    known_md = known['MD'].to_numpy(dtype=np.float32)
    known_z = known['Z'].to_numpy(dtype=np.float32)

    prefix_tw_gr = np.interp(known_tvt, tw_tvt, tw_gr)
    prefix_gr = gr_full.iloc[:mask_start].to_numpy(dtype=np.float32)
    prefix_residual = prefix_gr - prefix_tw_gr
    prefix_tw_rmse = float(np.sqrt(np.mean(prefix_residual ** 2)))
    prefix_tw_mae = float(np.mean(np.abs(prefix_residual)))

    last_known_tvt = float(last_known['TVT_input'])
    hidden_gr = hidden['GR'].to_numpy(dtype=np.float32)

    beam_strict = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 10, 40.0, 256.0, 2)
    beam_cons = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 10, 20.0, 144.0, 2)
    beam_loose = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 10, 8.0, 64.0, 2)
    beam_ultra = beam_predict(hidden_gr, tw_tvt, tw_gr, last_known_tvt, 15, 4.0, 32.0, 2)

    hidden_gr_filled = gr_full.iloc[mask_start:].to_numpy(dtype=np.float32)

    tw_mapped_gr_cons = np.interp(beam_cons, tw_tvt, tw_gr).astype(np.float32)
    tw_mapped_gr_loose = np.interp(beam_loose, tw_tvt, tw_gr).astype(np.float32)
    s_hidden_gr = pd.Series(hidden_gr_filled)
    tw_beam_corr_30 = s_hidden_gr.rolling(30, center=True, min_periods=5).corr(pd.Series(tw_mapped_gr_cons)).fillna(0.0).to_numpy(dtype=np.float32)
    tw_beam_corr_100 = s_hidden_gr.rolling(100, center=True, min_periods=10).corr(pd.Series(tw_mapped_gr_cons)).fillna(0.0).to_numpy(dtype=np.float32)

    offset_diffs = {
        f'tw_diff_{int(offset)}': hidden_gr_filled - np.float32(np.interp(last_known_tvt + float(offset), tw_tvt, tw_gr))
        for offset in OFFSETS
    }

    visible_shift = visible_gr_shift_fit(known_tvt, prefix_gr, tw_tvt, tw_gr)
    ncc_res, sc_ens = multi_scale_ncc(prefix_gr, known_tvt, hidden_gr_filled, hws=(8, 15, 25), stride=3)
    sc8, sc8_score = ncc_res[0]
    sc15, sc15_score = ncc_res[1]
    sc25, sc25_score = ncc_res[2]
    # Phase 5: Geology Plane Fit
    xy_full = df[["X", "Y"]].to_numpy(dtype=np.float64)
    self_wid_for_train = wid if is_train else None

    formations_full, knn_min_dist_full = formation_imputer.impute(xy_full, self_wid=self_wid_for_train)
    formations_post = formations_full[mask_start:]
    knn_min_dist_post = knn_min_dist_full[mask_start:]
    z_full = df["Z"].to_numpy(dtype=np.float32)
    z_post = hidden["Z"].to_numpy(dtype=np.float32)
    tvt_fs = {}
    for fi2, fn in enumerate(FORMATIONS):
        b_full, b_early, b_mid, b_late, b_wls = seg_b_well(known_tvt, known_z, formations_full[:mask_start, fi2]) if mask_start > 0 else (0.0, 0.0, 0.0, 0.0, 0.0)
        tvt_fs[f'tvtF_{fn}'] = (-z_post + formations_post[:, fi2] + b_full).astype(np.float32)
        tvt_fs[f'tvtFw_{fn}'] = (-z_post + formations_post[:, fi2] + b_wls).astype(np.float32)
        tvt_fs[f'tvtF50_{fn}'] = (-z_post + formations_post[:, fi2] + b_late).astype(np.float32)
        tvt_fs[f'bw_{fn}'] = np.float32(b_full)
        tvt_fs[f'bww_{fn}'] = np.float32(b_wls)
        tvt_fs[f'bw50_{fn}'] = np.float32(b_late)
        tvt_fs[f'bw_early_{fn}'] = np.float32(b_early)
        tvt_fs[f'bw_mid_{fn}'] = np.float32(b_mid)

    b_well_centroid = float(tvt_fs['bw_ANCC'])
    tvt_formula_pred_centroid = tvt_fs['tvtF_ANCC']

    # Row-level KNN
    row_ancc_full, row_ancc_std_full, row_min_dist_full = row_imputer.impute(xy_full, self_wid=self_wid_for_train)
    row_ancc_post = row_ancc_full[mask_start:]
    row_ancc_std_post = row_ancc_std_full[mask_start:]
    row_min_dist_post = row_min_dist_full[mask_start:]
    if mask_start > 0:
        b_per_row_row = known_tvt + known_z - row_ancc_full[:mask_start]
        b_well_row = calc_wls_b_well(b_per_row_row)
    else:
        b_well_row = 0.0
    tvt_formula_pred_row = -z_post + row_ancc_post + b_well_row

    dmd_safe = np.maximum(hidden['MD'] - float(last_known['MD']), 1e-5)

    # Particle Filters
    np.random.seed(42) # Deterministic particles
    pf_z_tvt, pf_z_std = run_pf_z_velocity(known, hidden, gr_roll5, tw_tvt, tw_gr)
    pf_ancc_tvt, pf_ancc_std = run_pf_ancc(known, hidden, tw_tvt, tw_gr)

    # Dense ANCC
    xy_eval = hidden[["X", "Y"]].to_numpy(dtype=np.float64)
    xy_known = known[["X", "Y"]].to_numpy(dtype=np.float64)
    dense_ancc_eval, dense_dist_eval, dense_std_eval = dense_imputer.impute(xy_eval, self_wid=self_wid_for_train)
    dense_ancc_known, _, _ = dense_imputer.impute(xy_known, self_wid=self_wid_for_train)

    if mask_start > 0:
        b_per_row_dense = known_tvt + known_z - dense_ancc_known
        b_well_dense = calc_wls_b_well(b_per_row_dense)
    else:
        b_well_dense = 0.0
    tvt_formula_pred_dense = -z_post + dense_ancc_eval + b_well_dense

    fft_freq, fft_pow = gr_fft_features(hidden_gr)

    features = pd.DataFrame({
        'well': wid,
        'prediction_id': [f'{wid}_{row_idx}' for row_idx in hidden.index],
        'row_idx': hidden.index.to_numpy(dtype=np.int32),
        'last_known_tvt': np.float32(last_known_tvt),
        'known_len': np.int32(mask_start),
        'hidden_len': np.int32(len(hidden)),
        'frac_hidden': ((hidden.index - mask_start) / max(len(hidden) - 1, 1)).astype(np.float32),
        'md': hidden['MD'].to_numpy(dtype=np.float32),
        'z': hidden['Z'].to_numpy(dtype=np.float32),
        'x': hidden['X'].to_numpy(dtype=np.float32),
        'y': hidden['Y'].to_numpy(dtype=np.float32),

        'gr': hidden_gr_filled,
        'gr_missing': hidden['GR'].isna().to_numpy(dtype=np.int8),
        'gr_roll5': gr_roll5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_roll21': gr_roll21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_grad': gr_grad.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std5': gr_std5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_std21': gr_std21.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag1': gr_lag1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead1': gr_lead1.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lag5': gr_lag5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_lead5': gr_lead5.iloc[mask_start:].to_numpy(dtype=np.float32),
        'gr_cumsum': (gr_cumsum.iloc[mask_start:] - gr_cumsum.iloc[mask_start - 1]).to_numpy(dtype=np.float32) if mask_start > 0 else gr_cumsum.iloc[mask_start:].to_numpy(dtype=np.float32),

        'dmd': (hidden['MD'] - float(last_known['MD'])).to_numpy(dtype=np.float32),
        'dz': (hidden['Z'] - float(last_known['Z'])).to_numpy(dtype=np.float32),
        'dx': (hidden['X'] - float(last_known['X'])).to_numpy(dtype=np.float32),
        'dy': (hidden['Y'] - float(last_known['Y'])).to_numpy(dtype=np.float32),

        'dx_dmd': ((hidden['X'] - float(last_known['X'])) / dmd_safe).to_numpy(dtype=np.float32),
        'dy_dmd': ((hidden['Y'] - float(last_known['Y'])) / dmd_safe).to_numpy(dtype=np.float32),
        'dz_dmd': ((hidden['Z'] - float(last_known['Z'])) / dmd_safe).to_numpy(dtype=np.float32),

        'dist_xy': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2).to_numpy(dtype=np.float32),
        'dist_xyz': np.sqrt((hidden['X'] - float(last_known['X'])) ** 2 + (hidden['Y'] - float(last_known['Y'])) ** 2 + (hidden['Z'] - float(last_known['Z'])) ** 2).to_numpy(dtype=np.float32),

        'prefix_tvt_step20': np.float32(recent_mean_diff(known_tvt, 20)),
        'prefix_tvt_step100': np.float32(recent_mean_diff(known_tvt, 100)),
        'prefix_tvt_md_slope100': np.float32(recent_slope(known_tvt, known_md, 100)),
        'prefix_tvt_z_slope100': np.float32(recent_slope(known_tvt, known_z, 100)),
        'prefix_tw_rmse': np.float32(prefix_tw_rmse),
        'prefix_tw_mae': np.float32(prefix_tw_mae),

        'beam_strict_delta': (beam_strict - np.float32(last_known_tvt)).astype(np.float32),
        'beam_cons_delta': (beam_cons - np.float32(last_known_tvt)).astype(np.float32),
        'beam_loose_delta': (beam_loose - np.float32(last_known_tvt)).astype(np.float32),
        'beam_ultra_delta': (beam_ultra - np.float32(last_known_tvt)).astype(np.float32),
        'beam_gap': (beam_loose - beam_cons).astype(np.float32),
        'beam_spread': (np.max([beam_strict, beam_cons, beam_loose, beam_ultra], axis=0) - np.min([beam_strict, beam_cons, beam_loose, beam_ultra], axis=0)).astype(np.float32),
        'beam_var': np.var([beam_strict, beam_cons, beam_loose, beam_ultra], axis=0).astype(np.float32),

        'tw_mapped_gr_cons': tw_mapped_gr_cons,
        'tw_mapped_gr_loose': tw_mapped_gr_loose,
        'tw_beam_cons_error': hidden_gr_filled - tw_mapped_gr_cons,
        'tw_beam_corr_30': tw_beam_corr_30,
        'tw_beam_corr_100': tw_beam_corr_100,

        'visible_gr_shift_ft': np.float32(visible_shift['visible_gr_shift_ft']),
        'visible_gr_shift_corr': np.float32(visible_shift['visible_gr_shift_corr']),
        'visible_gr_bias': np.float32(visible_shift['visible_gr_bias']),

        'ncc_shift_8': (sc8 - np.float32(last_known_tvt)).astype(np.float32),
        'ncc_corr_8': np.float32(sc8_score),
        'ncc_shift_15': (sc15 - np.float32(last_known_tvt)).astype(np.float32),
        'ncc_corr_15': np.float32(sc15_score),
        'ncc_shift_25': (sc25 - np.float32(last_known_tvt)).astype(np.float32),
        'ncc_corr_25': np.float32(sc25_score),
        'ncc_sc_ens': (sc_ens - np.float32(last_known_tvt)).astype(np.float32),

        'gr_fft_dom_freq': np.float32(fft_freq),
        'gr_fft_dom_power': np.float32(fft_pow),

        'pf_z_tvt_delta': (pf_z_tvt - np.float32(last_known_tvt)).astype(np.float32) if len(pf_z_tvt) else np.zeros(len(hidden), dtype=np.float32),
        'pf_z_std': pf_z_std.astype(np.float32) if len(pf_z_std) else np.zeros(len(hidden), dtype=np.float32),
        'pf_ancc_tvt_delta': (pf_ancc_tvt - np.float32(last_known_tvt)).astype(np.float32) if len(pf_ancc_tvt) else np.zeros(len(hidden), dtype=np.float32),
        'pf_ancc_std': pf_ancc_std.astype(np.float32) if len(pf_ancc_std) else np.zeros(len(hidden), dtype=np.float32),
        'dense_ancc': dense_ancc_eval.astype(np.float32),
        'dense_ancc_dz': (z_post - dense_ancc_eval).astype(np.float32),
        'dense_ancc_std': dense_std_eval.astype(np.float32),
        'dense_dist': dense_dist_eval.astype(np.float32),
        'dense_b_well': np.float32(b_well_dense),
        'dense_tvt_pred_delta': (tvt_formula_pred_dense - np.float32(last_known_tvt)).astype(np.float32),

        # Phase 5: Formation features
        "fk_b_well": np.float32(b_well_centroid),
        "fk_min_dist": knn_min_dist_post.astype(np.float32),
        "fk_tvt_formula_delta": (tvt_formula_pred_centroid - np.float32(last_known_tvt)).astype(np.float32),

        "knn_row_ANCC": row_ancc_post.astype(np.float32),
        "knn_row_ANCC_dz": (z_post - row_ancc_post).astype(np.float32),
        "knn_row_ANCC_std": row_ancc_std_post.astype(np.float32),
        "knn_row_dist": row_min_dist_post.astype(np.float32),
        "knn_row_b_well": np.float32(b_well_row),
        "knn_row_tvt_pred_delta": (tvt_formula_pred_row - np.float32(last_known_tvt)).astype(np.float32),
        "fk_vs_row_ANCC_diff": (formations_post[:, 0] - row_ancc_post).astype(np.float32),
    })

    for key, value in tvt_fs.items():
        if key.startswith('tvtF'):
            features[key] = (value - np.float32(last_known_tvt)).astype(np.float32)
        else:
            features[key] = value

    for name, values in offset_diffs.items():
        features[name] = values.astype(np.float32)

    if is_train:
        features['target'] = (hidden['TVT'].to_numpy(dtype=np.float32) - np.float32(last_known_tvt)).astype(np.float32)

    features.replace([np.inf, -np.inf], np.nan, inplace=True)
    return features

import multiprocessing

_FI = None
_RI = None
_DI = None

def _worker_init(fi, ri, di):
    global _FI, _RI, _DI
    _FI = fi
    _RI = ri
    _DI = di

def _build_wrapper(args):
    horizontal_path, typewell_path, is_train = args
    return build_hidden_features(horizontal_path, typewell_path, is_train, _FI, _RI, _DI)

def build_dataset(split_dir: Path, is_train: bool, formation_imputer, row_imputer, dense_imputer) -> pd.DataFrame:
    parts = []
    paths = sorted(split_dir.glob('*__horizontal_well.csv'))
    args = []
    for horizontal_path in paths:
        typewell_path = split_dir / f'{horizontal_path.name.split("__")[0]}__typewell.csv'
        args.append((horizontal_path, typewell_path, is_train))

    start = time.time()
    n_cpu = multiprocessing.cpu_count()
    print(f'Starting multiprocessing pool with {n_cpu} workers...')

    with multiprocessing.Pool(processes=n_cpu, initializer=_worker_init, initargs=(formation_imputer, row_imputer, dense_imputer)) as pool:
        for idx, res in enumerate(pool.imap_unordered(_build_wrapper, args), start=1):
            if res is not None:
                parts.append(res)
            if idx % 100 == 0 or idx == len(paths):
                print(f'processed {idx}/{len(paths)} wells in {time.time() - start:.1f}s')
    if not parts: raise RuntimeError(f'no usable wells found in {split_dir}')
    return pd.concat(parts, ignore_index=True)

In [12]:
# Modeling
def feature_columns(dataset: pd.DataFrame) -> list[str]:
    return [c for c in dataset.columns if c not in {'well', 'prediction_id', 'target'}]

def make_lgb_model(seed: int) -> LGBMRegressor:
    return LGBMRegressor(
        boosting_type="gbdt",
        learning_rate=0.0195,
        num_leaves=70,
        min_child_samples=11,
        min_child_weight=0.5,
        n_estimators=5000,
        n_jobs=-1,
        reg_alpha=97.97,
        reg_lambda=65.77,
        subsample=0.486,
        subsample_freq=1,
        colsample_bytree=0.542,
        objective="regression",
        metric="rmse",
        verbose=-1,
        device_type="gpu",
        gpu_use_dp=False,
        max_bin=255,
        random_state=seed
    )

def make_xgb_model(seed: int) -> XGBRegressor:
    return XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        learning_rate=0.06,
        max_depth=8,
        min_child_weight=10,
        subsample=0.7,
        colsample_bytree=0.85,
        reg_alpha=1.0,
        reg_lambda=20.0,
        tree_method="hist",
        device="cuda",
        n_jobs=-1,
        n_estimators=3000,
        early_stopping_rounds=200,
        random_state=seed
    )

def make_cb_model(seed: int) -> CatBoostRegressor:
    return CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=7,
        l2_leaf_reg=3.0,
        subsample=0.8,
        bootstrap_type="Poisson",
        random_seed=seed,
        task_type="GPU",
        eval_metric="RMSE",
        verbose=0,
        early_stopping_rounds=100
    )

def train_and_predict_cv(train_df: pd.DataFrame, test_df: pd.DataFrame) -> np.ndarray:
    predictors = feature_columns(train_df)

    # 3-Fold CV with Shuffled Groups to prevent Geographic Bias
    _unique_wells = np.unique(train_df['well'].values)
    _rng_cv = np.random.RandomState(SPLIT_SEED)
    _shuffled = _rng_cv.permutation(_unique_wells)
    _fold_map = {w: i % N_SPLITS for i, w in enumerate(_shuffled)}
    _fold_ids = np.array([_fold_map[w] for w in train_df['well'].values])

    folds = []
    for _f in range(N_SPLITS):
        tr_idx = np.where(_fold_ids != _f)[0]
        va_idx = np.where(_fold_ids == _f)[0]
        folds.append((tr_idx, va_idx))

    oof_preds_lgb = np.zeros(len(train_df))
    oof_preds_xgb = np.zeros(len(train_df))
    oof_preds_cb = np.zeros(len(train_df))

    test_preds_lgb = np.zeros((N_SPLITS, len(test_df)))
    test_preds_xgb = np.zeros((N_SPLITS, len(test_df)))
    test_preds_cb = np.zeros((N_SPLITS, len(test_df)))

    for fold, (train_idx, val_idx) in enumerate(folds):
        print(f"--- FOLD {fold+1}/{N_SPLITS} ---")
        X_tr, y_tr = train_df.iloc[train_idx][predictors], train_df.iloc[train_idx]['target']
        X_va, y_va = train_df.iloc[val_idx][predictors], train_df.iloc[val_idx]['target']

        # LGBM
        print("Training LGBM...")
        lgb = make_lgb_model(SPLIT_SEED + fold)
        lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[early_stopping(200, verbose=False)])
        oof_preds_lgb[val_idx] = lgb.predict(X_va)
        test_preds_lgb[fold] = lgb.predict(test_df[predictors])

        # XGB
        print("Training XGB...")
        xgb = make_xgb_model(SPLIT_SEED + fold)
        xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        oof_preds_xgb[val_idx] = xgb.predict(X_va)
        test_preds_xgb[fold] = xgb.predict(test_df[predictors])

        # CatBoost
        print("Training CatBoost...")
        cb = make_cb_model(SPLIT_SEED + fold)
        cb.fit(X_tr, y_tr, eval_set=(X_va, y_va), use_best_model=True)
        oof_preds_cb[val_idx] = cb.predict(X_va)
        test_preds_cb[fold] = cb.predict(test_df[predictors])

    print("--- RIDGE STACKING ---")
    meta_y = train_df['target'].to_numpy(dtype=np.float32)

    Sx = np.column_stack([oof_preds_lgb, oof_preds_xgb, oof_preds_cb])
    St = np.column_stack([np.mean(test_preds_lgb, axis=0), np.mean(test_preds_xgb, axis=0), np.mean(test_preds_cb, axis=0)])

    ridge = Ridge(alpha=1.0, fit_intercept=False, positive=True)
    ridge.fit(Sx, meta_y)
    oof_s = ridge.predict(Sx)
    test_s = ridge.predict(St)

    r_avg = np.sqrt(np.mean((meta_y - Sx.mean(1))**2))
    r_stk = np.sqrt(np.mean((meta_y - oof_s)**2))

    wts = ridge.coef_ / max(ridge.coef_.sum(), 1e-9)
    print(f"Avg RMSE: {r_avg:.4f} | Ridge RMSE: {r_stk:.4f}")
    print(f"Weights: LGB={wts[0]:.3f}, XGB={wts[1]:.3f}, CB={wts[2]:.3f}")

    final_oof = oof_s if r_stk < r_avg else Sx.mean(1)
    final_test = test_s if r_stk < r_avg else St.mean(1)

    base = train_df['last_known_tvt'].to_numpy(dtype=np.float32)
    ytrue = meta_y + base
    pf_oof = train_df['pf_ancc_tvt_delta'].to_numpy(dtype=np.float32)

    print("Grid search alpha * tau * w_pf...")
    best_cfg, best_r = (1.0, None, 0.0), np.inf
    for alpha in np.arange(0.65, 1.01, 0.05):
        for tau in [None, 25., 50., 100., 200., 350.]:
            for w_pf in [0.0, 0.05, 0.10, 0.15]:
                d = final_oof * (1 - w_pf) + pf_oof * w_pf
                if tau is not None:
                    d *= (1. - np.exp(-np.maximum(train_df['dmd'].to_numpy(dtype=np.float32), 0.) / tau))
                d *= alpha
                r = np.sqrt(np.mean((ytrue - (base + d))**2))
                if r < best_r:
                    best_r, best_cfg = r, (alpha, tau, w_pf)

    ALPHA, TAU, W_PF = best_cfg
    print(f"Best: alpha={ALPHA:.2f} tau={TAU} w_pf={W_PF:.2f} | abs TVT RMSE={best_r:.4f}")

    pf_test = test_df['pf_ancc_tvt_delta'].to_numpy(dtype=np.float32)
    test_d = final_test * (1 - W_PF) + pf_test * W_PF
    if TAU is not None:
        test_d *= (1. - np.exp(-np.maximum(test_df['dmd'].to_numpy(dtype=np.float32), 0.) / TAU))
    test_d *= ALPHA

    test_pred_tvt = test_d + test_df['last_known_tvt'].to_numpy(dtype=np.float32)

    # SavGol smoothing
    def sg_smooth(df, pred_vals, sg_w=17, sg_p=3):
        out = pred_vals.copy()
        for well, g in df.groupby('well', sort=False):
            v = out[g.index]
            n = len(v)
            wl = min(sg_w, n)
            if wl % 2 == 0: wl -= 1
            if wl >= sg_p + 2:
                v = savgol_filter(v, wl, sg_p)
            out[g.index] = v
        return out

    test_pred_tvt = sg_smooth(test_df, test_pred_tvt)

    return test_pred_tvt

In [13]:
# Main Execution
train_dir = DATA_DIR / 'train'
test_dir = DATA_DIR / 'test'
train_paths = sorted(train_dir.glob('*__horizontal_well.csv'))

print('>> build PLANE-FIT formation imputer')
formation_imputer = FormationPlaneKNN(train_paths)
print('>> build row-level (X, Y) ANCC imputer')
row_imputer = RowKNN(train_paths)
print('>> build dense (X, Y) ANCC imputer')
dense_imputer = DenseANCCImputer(train_paths)

print('>> building train features')
train_dataset = build_dataset(train_dir, True, formation_imputer, row_imputer, dense_imputer)
print(f'train rows: {len(train_dataset):,}')

print('>> building test features')
test_dataset = build_dataset(test_dir, False, formation_imputer, row_imputer, dense_imputer)
print(f'test rows: {len(test_dataset):,}')

predicted_tvt = train_and_predict_cv(train_dataset, test_dataset)

submission = pd.DataFrame({
    'id': test_dataset['prediction_id'],
    'tvt': predicted_tvt.astype(np.float32),
})

>> build PLANE-FIT formation imputer


KeyError: 'wid'

In [ ]:
# Exact-Overlap Replacement
print('>> Applying Exact-Overlap Replacement...')
train_dfs = []
for p in train_paths:
    try:
        df = pd.read_csv(p, usecols=['X', 'Y', 'Z', 'TVT'])
        df = df[df['TVT'].notna()]
        train_dfs.append(df)
    except Exception: continue
if train_dfs:
    train_all = pd.concat(train_dfs, ignore_index=True)
    train_all['X_r'] = train_all['X'].round(2)
    train_all['Y_r'] = train_all['Y'].round(2)
    train_all['Z_r'] = train_all['Z'].round(2)
    train_dict = train_all.drop_duplicates(subset=['X_r', 'Y_r', 'Z_r']).set_index(['X_r', 'Y_r', 'Z_r'])['TVT'].to_dict()

    replaced = 0
    test_xs = test_dataset['x'].to_numpy()
    test_ys = test_dataset['y'].to_numpy()
    test_zs = test_dataset['z'].to_numpy()
    for i in range(len(submission)):
        key = (round(test_xs[i], 2), round(test_ys[i], 2), round(test_zs[i], 2))
        if key in train_dict:
            submission.loc[i, 'tvt'] = train_dict[key]
            replaced += 1
    print(f"Replaced {replaced}/{len(submission)} test predictions with exact known training overlap.")

submission.to_csv(OUTPUT_PATH, index=False)
print(f'wrote {len(submission):,} predictions to {OUTPUT_PATH}')

